# Moving Average Family

In a moving average (MA) process, the current value depends linearly on the mean of the series, the current error term, and past error terms. The moving average model is denoted as MA(q), where q is the order.

**Table of contents**<a id='toc0_'></a>    
- [Preprocessing](#toc1_)    
  - [Import + setup](#toc1_1_)    
  - [Upload](#toc1_2_)    
  - [Make time serie stationary](#toc1_3_)    
  - [Plotting time series + ACF + PACF](#toc1_4_)    
  - [First main](#toc1_5_)    
  - [Check autocorrelation](#toc1_6_)    
- [Forecast](#toc2_)    
  - [Rolling forecast](#toc2_1_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc1_'></a>[Preprocessing](#toc0_)

## <a id='toc1_1_'></a>[Import + setup](#toc0_)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from src.config import set_seeds
import src.pipeline as pipe
import src.models as mod
import src.evaluation as eval
import src.reporting as rep
import src.visualization as visual
import pandas as pd
import time
import warnings
warnings.filterwarnings("ignore", message="'force_all_finite' was renamed")
warnings.filterwarnings("ignore", message="Non-stationary starting moving average parameters")
warnings.filterwarnings("ignore", category=UserWarning, module="statsmodels")

In [3]:
set_seeds()

Random seeds set to 42. Deterministic operations enabled.


## <a id='toc1_2_'></a>[Upload](#toc0_)

In [4]:
df = pipe.load_data()
display(df.head())

Dropped 9 unusable indicators.
Dataset loaded: 65 years (from 1960 to 2024), 27 variables.


,population_percent,population_growth,population_abs,employment_tot,employment_male,employment_female,forestarea_percent,forestarea_abs,agriland_percent,agriland_abs,...,fertilizer_percent,livestock_production_index,food_production_index,crop_production_index,cereal_production,cerealyield_abs,valueadded_percent,valueadded_dollars,exports_percent,imports_percent
1960-01-01,40.639,NaN,20400656.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1961-01-01,40.144,-0.557139,20287312.0,NaN,NaN,NaN,NaN,NaN,70.324028,206830.0,...,72.707716,70.91,85.93,93.68,13933400.0,2181.5,NaN,NaN,NaN,NaN
1962-01-01,39.645,-0.574190,20171158.0,NaN,NaN,NaN,NaN,NaN,70.218626,206520.0,...,70.071359,72.59,86.90,94.43,14433210.0,2225.3,NaN,NaN,2.563660,16.566057
1963-01-01,39.147,-0.534554,20063620.0,NaN,NaN,NaN,NaN,NaN,69.735813,205100.0,...,63.883735,65.95,86.37,97.19,13324660.0,2115.2,NaN,NaN,2.651714,14.571604
1964-01-01,38.650,-0.455075,19972523.0,NaN,NaN,NaN,NaN,NaN,69.572609,204620.0,...,64.593354,69.57,90.28,101.31,14007520.0,2243.0,NaN,NaN,2.792923,15.115329


## Sanity Check

In [5]:
visual.plot_sanity_check(df)

Check plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\00_DIFFCHECK\DIFFCHECK_population_percent.png
Check plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\00_DIFFCHECK\DIFFCHECK_population_growth.png
Check plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\00_DIFFCHECK\DIFFCHECK_population_abs.png
Check plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\00_DIFFCHECK\DIFFCHECK_employment_tot.png
Check plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\00_DIFFCHECK\DIFFCHECK_employment_male.png
Check plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\00_DIFFCHECK\DIFFCHECK_employment_female.png
Check plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\00_DIFFCHECK\DIFFCHECK_forestarea_percent.png
Check plot saved: C:\Users\oldan\Desktop\RuralDevelopm

## Finding correct Integration Order for each indicator
Searching d on both the train set and the full serie to avoid data leakage. <br>
Saving all necessary information in master configuration (excel)

In [5]:
config_data = []

print("--- Integration Order and Initialization Parameters Configuration ---")
# i search for the optimal d differentiations on the train set and on the full series 
# to apply as parameter d of the ARIMA order in the following steps
for col in df.columns:
    if df[col].dropna().empty: continue
    full_series = df[col].dropna()
    
    train, test = pipe.split_train_test(df, col)
    train_series = train['Value'].dropna()
    
    print(f"\nProcessing {col}...")
    
    # VALIDATION PHASE (Train Set)
    # to avoid data leakage I calculate the integration order on the train serie
    print("Validation phase")
    d_train = pipe.find_integration_order(train_series, max_d=2)
    # to reconstruct the predictions on the test set, we save train set's last value
    # real_pred_t1 = last_train_val + pred_diff_t1
    last_val_train = train_series.iloc[-1]
    
    # FUTURE PHASE (Full Set)
    # I calculate the integration order on the full history to forecast values up until 2030
    print("Future phase")
    d_full = pipe.find_integration_order(full_series, max_d=1)
    # to reconstruct the predictions on future, we save full history's last value
    last_val_full = full_series.iloc[-1]
    
    config_data.append({
        'Indicator': col,
        
        # VALIDATION (Test Set)
        'd_train': d_train,
        'last_val_train': last_val_train,
        'train_end_year': train_series.index.max().year,
        
        # FUTURE (2030)
        'd_full': d_full,
        'last_val_full': last_val_full,
        'full_end_year': full_series.index.max().year
    })

config_df = pd.DataFrame(config_data)
config_df.set_index('Indicator', inplace=True)

rep.save_master_config(config_df)

--- Integration Order and Initialization Parameters Configuration ---

Processing population_percent...
Validation phase
  StdDev (d=0): 2.3230
  StdDev (d=1): 0.1811
  StdDev (d=2): 0.0386
  -> Optimal d (Variance Rule): 2
  -> Optimal d (KPSS Test):     2
  => FINAL DECISION: d=2
Future phase
  StdDev (d=0): 2.8435
  StdDev (d=1): 0.1685
  StdDev (d=2): 0.0388
  -> Optimal d (Variance Rule): 2
  -> Optimal d (KPSS Test):     1
  => FINAL DECISION: d=1

Processing population_growth...
Validation phase
  StdDev (d=0): 0.3088
  StdDev (d=1): 0.1528
  StdDev (d=2): 0.1821
  -> Optimal d (Variance Rule): 1
  -> Optimal d (KPSS Test):     1
  => FINAL DECISION: d=1
Future phase
  StdDev (d=0): 0.5099
  StdDev (d=1): 0.2100
  StdDev (d=2): 0.2974
  -> Optimal d (Variance Rule): 1
  -> Optimal d (KPSS Test):     0
  => FINAL DECISION: d=1

Processing population_abs...
Validation phase
  StdDev (d=0): 462571.8079
  StdDev (d=1): 59917.6575
  StdDev (d=2): 28938.4882
  -> Optimal d (Variance R

## Grid search to find best q
I run the rolling MA forecast on test set's fitting, in order to find q parameter that results in the smallest RMSE. <br>
I save the best q found for each indicator in a dedicated DataFrame i will be able to access in the following cells.


In [6]:
Q_CANDIDATES = [1, 2, 3, 4] 

best_qs = {}
print("STARTING GRID SEARCH")
for col in df.columns:
    if df[col].dropna().empty: continue
    d_train = int(config_df.loc[col]['d_train'])
    
    train, test = pipe.split_train_test(df, col)
    train_serie = train['Value'].dropna()
    test_serie = test['Value'].dropna()
    
    best_q = 1
    best_rmse = float('inf')
    
    for q in Q_CANDIDATES:
        try:
            pred = mod.predict_arima_rolling(
                train_data=train_serie, 
                test_data=test_serie, 
                order=(0, d_train, q),
                refit=True
            )
            metrics = eval.compute_errors(test_serie, pred)
            
            if metrics['RMSE'] < best_rmse:
                best_rmse = metrics['RMSE']
                best_q = q
        except:
            continue
            
    print(f"{col} -> Best q: {best_q} (RMSE: {best_rmse:.4f})")
    best_qs[col] = best_q

STARTING GRID SEARCH
Starting Rolling Forecast ARIMA(0, 2, 1) over 13 steps...
Starting Rolling Forecast ARIMA(0, 2, 2) over 13 steps...
Starting Rolling Forecast ARIMA(0, 2, 3) over 13 steps...
Starting Rolling Forecast ARIMA(0, 2, 4) over 13 steps...
population_percent -> Best q: 1 (RMSE: 0.0344)
Starting Rolling Forecast ARIMA(0, 1, 1) over 13 steps...
Starting Rolling Forecast ARIMA(0, 1, 2) over 13 steps...
Starting Rolling Forecast ARIMA(0, 1, 3) over 13 steps...
Starting Rolling Forecast ARIMA(0, 1, 4) over 13 steps...
population_growth -> Best q: 1 (RMSE: 0.3992)
Starting Rolling Forecast ARIMA(0, 2, 1) over 13 steps...
Starting Rolling Forecast ARIMA(0, 2, 2) over 13 steps...
Starting Rolling Forecast ARIMA(0, 2, 3) over 13 steps...
Starting Rolling Forecast ARIMA(0, 2, 4) over 13 steps...
population_abs -> Best q: 1 (RMSE: 62611.0923)
Starting Rolling Forecast ARIMA(0, 1, 1) over 7 steps...
Starting Rolling Forecast ARIMA(0, 1, 2) over 7 steps...
Starting Rolling Forecast ARI

## Validation phase on test set
I train a rolling MA model on the train set using (0, d, q) order, where d is the number of best differentiations found on the train set and q is the best q found in the prior cell. I then predict the values of the test set, compute the errors, plot a comparison with the real values and a random walk with drift baseline and i report all of the data i get on the different log files

In [7]:
BASELINE_METHOD = 'drift'

print("VALIDATION PHASE")
for col in df.columns:
    if col not in best_qs: continue
    best_q = best_qs[col]
    d_train = int(config_df.loc[col]['d_train'])
    
    train, test = pipe.split_train_test(df, col)
    train_serie = train['Value'].dropna()
    test_serie = test['Value'].dropna()
    
    start_time = time.time()
    final_pred = mod.predict_arima_rolling(
        train_data=train_serie,
        test_data=test_serie,
        order=(0, d_train, best_q)
    )
    elapsed_time = time.time() - start_time
    
    baseline_pred = mod.get_baseline_prediction(
        BASELINE_METHOD, 
        train_serie, 
        test_serie.index
    )
    
    pred_metrics = eval.compute_errors(test_serie, final_pred)
    pred_residuals = eval.compute_residual_diagnostics(test_serie, final_pred)
    
    print(f"  -> Residuals: Mean={pred_residuals['residual_mean']:.4f}, "
        f"Shapiro P={pred_residuals['shapiro_wilk_pvalue']:.4f}, "
        f"Ljung-Box P={pred_residuals['ljung_box_pvalue']:.4f}")
    
    visual.plot_forecast(
        train=train,
        test=test,
        variable_name=col,
        model_name="MA",
        folder_name="02_MA",
        prediction=final_pred,
        baseline=baseline_pred,
        baseline_name=BASELINE_METHOD,
        rmse=metrics['RMSE'],
        save_plot=True
    )
    
    try:
        lb_p = pred_residuals['ljung_box_pvalue']
        sw_p = pred_residuals['shapiro_wilk_pvalue']
        
        lb_status = "OK" if lb_p > 0.05 else "NO"
        sw_status = "OK" if sw_p > 0.05 else "NO"
        plot_title_suffix = f"\nLjung-Box p={lb_p} ({lb_status}) | Shapiro p={sw_p} ({sw_status})"
        
        visual.plot_residuals(
            y_true=test_serie,
            y_pred=final_pred,
            variable_name=col,
            model_name=f"MA",
            folder_name="02_residMA"
        )
    except Exception as e:
        print(f"Residual plot failed: {e}")
    
    rep.save_experiment_results(
        indicator=col,
        model_name='MA',
        configuration=f'order=(0, {d_train}, {best_q})',
        y_test=test_serie,
        y_pred=final_pred,
        years_test=test_serie.index.year,
        y_train=train_serie,
        params={'p':0, 'd':d_train, 'q':best_q},
        training_time=elapsed_time
    )

VALIDATION PHASE
Starting Rolling Forecast ARIMA(0, 2, 1) over 13 steps...
  -> Residuals: Mean=-0.0108, Shapiro P=0.0000, Ljung-Box P=0.9824
Plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\02_MA\MA_population_percent.png
Residuals saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\02_residMA\residMA_population_percent.png
Saving results for MA | population_percent...
Leaderboard updated: population_percent | MA
Save leaderboard complete.
Starting Rolling Forecast ARIMA(0, 1, 1) over 13 steps...
  -> Residuals: Mean=-0.0603, Shapiro P=0.0080, Ljung-Box P=0.1495
Plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\02_MA\MA_population_growth.png
Residuals saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\02_residMA\residMA_population_growth.png
Saving results for MA | population_growth...
Leaderboard updated: population_growth | MA
Save leaderboard complete.
Sta

## Future forecasts

In [8]:
TARGET_YEAR = 2030
BASELINE_METHOD = 'drift'

print(f"Future forecasting (up to {TARGET_YEAR}) ---")
for col in df.columns:
    if col not in best_qs: continue
    
    best_q = best_qs[col]
    d_full = int(config_df.loc[col]['d_full'])
    
    full_series = df[col].dropna()
    
    last_date = full_series.index[-1]
    if last_date.year >= TARGET_YEAR:
        print(f"  -> Data already available until {last_date.year}. Skipping.")
        continue
    future_index = pd.date_range(
        start=last_date + pd.DateOffset(years=1),
        end=pd.Timestamp(f"{TARGET_YEAR}-01-01"),
        freq='YS'
    )
    
    start_time = time.time()
    future_pred = mod.predict_arima_family(
        train_data=full_series,
        forecast_index=future_index,
        order=(0, d_full, best_q)
    )
    elapsed_time = time.time() - start_time
    future_df = future_pred.to_frame(name='pred')
    future_df['year'] = future_df.index.year
    future_df.reset_index(drop=True, inplace=True)
    
    baseline_pred = mod.get_baseline_prediction(
        method_name=BASELINE_METHOD, 
        train_data=full_series,
        forecast_index=future_index
    )
    baseline_df = baseline_pred.to_frame(name='pred')
    baseline_df['year'] = baseline_df.index.year
    baseline_df.reset_index(drop=True, inplace=True)
    
    visual.plot_future_forecasts(
        full_history=full_series,
        future_pred=future_df,
        baseline_pred=baseline_df,
        variable_name=col,
        model_name='MA',
        baseline_name=BASELINE_METHOD,
        folder_name='02_futureMA',
        save_plots=True
    )

Future forecasting (up to 2030) ---
Future Plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\02_futureMA\futureMA_population_percent.png
Future Plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\02_futureMA\futureMA_population_growth.png
Future Plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\02_futureMA\futureMA_population_abs.png
Future Plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\02_futureMA\futureMA_employment_tot.png
Future Plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\02_futureMA\futureMA_employment_male.png
Future Plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\02_futureMA\futureMA_employment_female.png
Future Plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\02_futureMA\futureMA_forestarea_percent.png
Future Plot saved: C:\Use

==================================================================================

<a class="anchor" id="autocorrelation"> </a>
## <a id='toc1_6_'></a>[Check autocorrelation](#toc0_)

suitable indicators are the one that:
- in the ACF plot --> shows cut-off behaviour: significant coefficients up until lag q, then abruptly becomes non significant.
- in the PACF plot --> decrease gradually <br>
q will become the order of our MA(q) model

[Box-jenkins](https://openforecast.org/adam/BJApproach.html) <br>
For an MA(q) process, ACF will drop abruptly right after the lag q; <br>
For an MA(q) process, PACF will decline either exponentially or alternatingly (based on the specific values of parameters), starting from the lag q. <br>
However, these rules are not necessarily bi-directional and might not work in practice, e.g. if we deal with MA(q), ACF drops abruptly right after the lag q, but if ACF drops abruptly after the lag q, then this does not necessarily mean that we deal with MA(q). The former follows directly from the assumed “true” model, while the latter refers to the identification of the model on the data, and there can be different reasons for the ACF to behave in the way it does.

L'ispezione visiva mostra un taglio netto in entrambi i grafici, suggerendo un processo a memoria breve o parametri deboli. Nessuna time serie è perfettamente adatta a MA ma provo a farla andare su time serie che mostrano ACF abruptly decreasing

In [10]:
SUITABLE = ["Rural population growth (annual %)",
"Rural population",
"Agriculture, forestry, and fishing, value added (% of GDP)",
"Agriculture, forestry, and fishing, value added (current US$)",
"Annual freshwater withdrawals, agriculture (% of total freshwater withdrawal)",
"Cereal yield (kg per hectare)",
"Livestock production index (2014-2016 = 100)",
"Food production index (2014-2016 = 100)",
"Crop production index (2014-2016 = 100)",
"Cereal production (metric tons)",
"Permanent cropland (% of land area)",
"Land under cereal production (hectares)",
"Arable land (% of land area)",
"Arable land (hectares per person)",
"Arable land (hectares)",
"Agricultural land (% of land area)",
"Agricultural land (sq. km)",
"Fertilizer consumption (kilograms per hectare of arable land)",
"Fertilizer consumption (% of fertilizer production)"
]

In [ ]:
# def rolling_forecast(df: pd.DataFrame, train_len: int, horizon: int, window: int, method: str) -> list:
#     total_len = train_len + horizon

#     if method == 'mean':
#         pred_mean = []
#         for i in range(train_len, total_len, window):
#             mean = np.mean(df["Diff"][:i])
#             pred_mean.extend([mean] * window)
#         return pred_mean

#     elif method == 'last':
#         pred_last_value = []
#         for i in range(train_len, total_len, window):
#             last_value = df["Diff"][:i].iloc[-1]
#             pred_last_value.extend([last_value] * window)
#         return pred_last_value

#     elif method == 'MA':
#         pred_MA = []
#         for i in range(train_len, total_len, window):
#             if len(df["Diff"][:i]) < 12:
#                 pred_MA.extend([np.nan] * window)
#                 continue
#             try:
#                 model = SARIMAX(df["Diff"][:i].values, order=(0,0,1))
#                 res = model.fit(disp=False)
#                 predictions = res.get_prediction(start=i, end=i + window - 1)
#                 oos_pred = predictions.predicted_mean
#                 pred_MA.extend(oos_pred)
#             except Exception as e:
#                 print(f"Errore AR per finestra {i}: {e}")
#                 pred_MA.extend([np.nan] * window)
#         return pred_MA

In [ ]:
# # ROLLING FORECAST TO MODEL MOVING AVERAGE
# from statsmodels.tsa.statespace.sarimax import SARIMAX

# def rolling_forecast(df: pd.DataFrame, train_len: int, horizon: int, window: int, method: str) -> list:
#     total_len = train_len + horizon
#     if method == 'mean':
#         pred_mean = []
#         for i in range(train_len, total_len, window):
#             mean = np.mean(df[:i])
#             pred_mean.extend(mean for _ in range(window))
#         return pred_mean

#     elif method == 'last':
#         pred_last_value = []
#         for i in range(train_len, total_len, window):
#             last_value = df[:i].iloc[-1]
#             pred_last_value.extend(last_value for _ in range(window))
#         return pred_last_value
    
#     elif method == 'MA':
#         pred_MA = []
#         for i in range(train_len, total_len, window):
#             model = SARIMAX(df[:i], order=(0,0,2))
#             res = model.fit(disp=False)
#             predictions = res.get_prediction(0, i + window - 1)
#             oos_pred = predictions.predicted_mean.iloc[-window:]
#             pred_MA.extend(oos_pred)
#         return pred_MA
# pred_df = test.copy()

# TRAIN_LEN = len(train)
# HORIZON = len(test)
# WINDOW = 1

# pred_mean = rolling_forecast(df_livestock['Diff'], TRAIN_LEN, HORIZON, WINDOW, 'mean')
# pred_last_value = rolling_forecast(df_livestock['Diff'], TRAIN_LEN, HORIZON, WINDOW, 'last')
# pred_MA = rolling_forecast(df_livestock['Diff'], TRAIN_LEN, HORIZON, WINDOW, 'MA')

# pred_df['pred_mean'] = pred_mean
# pred_df['pred_last_value'] = pred_last_value
# pred_df['pred_MA'] = pred_MA

# pred_df.head()

,Year,Value,Diff,pred_mean,pred_last_value,pred_MA
55,2016,102.47,0.96,0.566667,5.49,0.503091
56,2017,100.09,-2.38,0.573818,0.96,-0.494642
57,2018,106.33,6.24,0.521071,-2.38,0.063690
58,2019,107.13,0.80,0.621404,6.24,-0.219083
59,2020,104.73,-2.40,0.624483,0.80,-0.785769
